# B-01: Forecasting baselines + ML benchmark (FC-01, D-36/D-37)

First forecasting benchmark (`05_benchmarking`). **Driver-conditional, direct multi-horizon**, partial-pooled.
Primary data = **External SMS/MET** (D-35) via `data/Hourly/forecast_features.csv`.

- **Track A (hourly):** horizons {1,6,12,24,48} h.  **Track B (daily-mean):** horizons {1,3,7,14} d.
- **Leak-free:** features = AR lags of gap-filled CH4 + lagged-only FCO2/GPP/Reco (origin t) + **future exog** (met/soil/
  livestock/mgmt at t+h, supplied) + calendar + tower dummies. FCO2/GPP/Reco never used at t+h.
- **Train on gap-filled, evaluate on OBSERVED only.**
- **CV:** Towers 4/9 train(target≤2021)/test 2022–2023; **Tower 2 expanding-window rolling-origin** within 2017–2019
  (2 folds, donor = Tower 4). Metrics: RMSE/MAE/R²/MBE + **skill vs persistence & climatology**. (T2 R² unstable →
  lead with RMSE/skill.)


In [1]:
from pathlib import Path
import datetime, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
ALGOS=["RF","XGB"]
HORIZONS={"A":[1,6,12,24,48],"B":[1,3,7,14]}
T2_FOLDS=[("2018-06-30","2018-07-01","2018-12-31"),("2018-12-31","2019-01-01","2019-05-31")]

mat=pd.read_csv(HOURLY/"forecast_features.csv",low_memory=False)
mat["Datetime"]=pd.to_datetime(mat["Datetime"],format="mixed")
AR_A=[c for c in mat.columns if c.startswith("ar_")]
FX=[c for c in mat.columns if c.startswith("fx")]
DUM=["is_t2","is_t4","is_t9"]
print("rows",len(mat),"| AR_A",len(AR_A),"| FX",len(FX))

rows 210459 | AR_A 20 | FX 19


## 1  Helpers (model fit, metrics, climatology, aligned frames)

In [2]:
def fit_model(algo, tr, feat):
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    if algo=="RF":
        m=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    else:
        m=XGBRegressor(n_estimators=500,max_depth=6,learning_rate=0.05,subsample=0.8,
                       colsample_bytree=0.8,n_jobs=-1,random_state=42)
    m.fit(Xi,tr["target"].values); return m,imp

def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if (len(y)>1 and np.var(y)>0) else np.nan
    return rmse(y,p), float(mean_absolute_error(y,p)), r2, float(np.mean(p-y))

def climatology(tr, te, unit):
    t=te["tower"].iloc[0]; trt=tr[tr.tower==t]
    if len(trt)<24: trt=tr
    gl=float(trt["target"].mean())
    if unit=="h":
        g=trt.assign(mo=trt.ttime.dt.month,hr=trt.ttime.dt.hour).groupby(["mo","hr"])["target"].mean()
        keys=list(zip(te.ttime.dt.month,te.ttime.dt.hour)); return np.array([g.get(k,gl) for k in keys])
    g=trt.assign(mo=trt.ttime.dt.month).groupby("mo")["target"].mean()
    return np.array([g.get(m,gl) for m in te.ttime.dt.month])

def build_hourly_tables():
    T={}
    for t in [2,4,9]:
        T[t]=mat[mat.tower==t].set_index("Datetime").sort_index()
    return T, AR_A

def build_daily_tables():
    AR_B=[f"ar_ch4_dlag{L}" for L in [1,2,3,7,14]]+["ar_ch4_drm7"]
    T={}
    for t in [2,4,9]:
        df=mat[mat.tower==t].set_index("Datetime").sort_index()
        cnt=df["y_observed"].resample("D").count()
        yo=df["y_observed"].resample("D").mean().where(cnt>=6)
        yg=df["y_gapfilled"].resample("D").mean()
        dd=pd.DataFrame(index=yg.index); dd["y_observed"]=yo; dd["y_gapfilled"]=yg
        for L in [1,2,3,7,14]: dd[f"ar_ch4_dlag{L}"]=yg.shift(L)
        dd["ar_ch4_drm7"]=yg.shift(1).rolling(7,min_periods=1).mean()
        for c in FX: dd[c]=df[c].resample("D").mean()
        for tt in [2,4,9]: dd[f"is_t{tt}"]=1.0 if tt==t else 0.0
        T[t]=dd
    return T, AR_B

def aligned(T, h, unit, ar_cols):
    parts=[]
    off=pd.Timedelta(hours=h) if unit=="h" else pd.Timedelta(days=h)
    for t,df in T.items():
        f=df[ar_cols+DUM].copy()
        for c in FX: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["persist"]=df["y_gapfilled"]
        f["tower"]=t; f["ttime"]=df.index+off
        parts.append(f)
    return pd.concat(parts)

## 2  Run one horizon (MAIN towers 4/9 single split + Tower 2 expanding folds)

In [3]:
def emit(track,h,t,bydict):
    rp=rmse(*bydict["persist"]); rc=rmse(*bydict["clim"]); rows=[]
    for model,(y,p) in bydict.items():
        r,mae,r2,mbe=metrics(y,p)
        rows.append(dict(track=track,horizon=h,tower=t,model=model,RMSE=round(r,3),MAE=round(mae,3),
            R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),n_test=len(y),
            skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            skill_clim=round(1-r/rc,3) if rc>0 else np.nan))
    return rows

def run_horizon(track, h, unit, ar_cols, T):
    feat=ar_cols+FX+DUM; A=aligned(T,h,unit,ar_cols); rows=[]
    # MAIN: Towers 4 & 9 (train target<=2021, test 2022-2023)
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    fitted={a:fit_model(a,tr,feat) for a in ALGOS}
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        bd={"persist":(te["target"].values,te["persist"].values),
            "clim":(te["target"].values,climatology(tr,te,unit))}
        for a in ALGOS:
            m,imp=fitted[a]; bd[a]=(te["target"].values,m.predict(imp.transform(te[feat].values)))
        rows+=emit(track,h,t,bd)
    # Tower 2: expanding-window folds within 2017-2019 (donor = Tower 4)
    acc={k:([],[]) for k in ["persist","clim"]+ALGOS}
    for cut,s,e in T2_FOLDS:
        trf=A[(A.ttime<=pd.Timestamp(cut)) & A.target.notna()]
        te=A[(A.tower==2)&(A.ttime>=pd.Timestamp(s))&(A.ttime<=pd.Timestamp(e))&A.target.notna()]
        if len(te)<5 or len(trf)<50: continue
        y=te["target"].values
        acc["persist"][0].append(y); acc["persist"][1].append(te["persist"].values)
        acc["clim"][0].append(y); acc["clim"][1].append(climatology(trf,te,unit))
        for a in ALGOS:
            m,imp=fit_model(a,trf,feat); acc[a][0].append(y); acc[a][1].append(m.predict(imp.transform(te[feat].values)))
    if acc["persist"][0]:
        bd={k:(np.concatenate(v[0]),np.concatenate(v[1])) for k,v in acc.items()}
        rows+=emit(track,h,2,bd)
    return rows

## 3  Run both tracks, all horizons

In [4]:
TA,AR_A_=build_hourly_tables()
TB,AR_B_=build_daily_tables()
ALL=[]
for h in HORIZONS["A"]:
    ALL+=run_horizon("A",h,"h",AR_A_,TA); print("Track A h=",h,"done",flush=True)
for h in HORIZONS["B"]:
    ALL+=run_horizon("B",h,"D",AR_B_,TB); print("Track B d=",h,"done",flush=True)
R=pd.DataFrame(ALL); print("rows",len(R))

Track A h= 1 done
Track A h= 6 done
Track A h= 12 done
Track A h= 24 done
Track A h= 48 done
Track B d= 1 done
Track B d= 3 done
Track B d= 7 done
Track B d= 14 done
rows 108


## 4  Results — skill vs persistence (the headline) and per-horizon tables

In [5]:
pd.set_option("display.width",200)
print("=== Track A (hourly) — RMSE and skill-vs-persistence ===")
for t in [4,9,2]:
    sub=R[(R.track=="A")&(R.tower==t)]
    if sub.empty: continue
    piv=sub.pivot_table(index="model",columns="horizon",values="RMSE")
    sk =sub.pivot_table(index="model",columns="horizon",values="skill_persist")
    print(f"\n-- Tower {t} | RMSE --"); print(piv.round(2).to_string())
    print(f"-- Tower {t} | skill vs persistence (RF/XGB>0 = beats persistence) --"); print(sk.round(3).to_string())
print("\n=== Track B (daily) — RMSE ===")
for t in [4,9,2]:
    sub=R[(R.track=="B")&(R.tower==t)]
    if sub.empty: continue
    print(f"\n-- Tower {t} --"); print(sub.pivot_table(index="model",columns="horizon",values="RMSE").round(2).to_string())
print("\n=== R2 (towers 4/9; T2 unstable/omitted) ===")
print(R[(R.tower.isin([4,9]))&(R.model.isin(["RF","XGB"]))].pivot_table(index=["track","tower","model"],columns="horizon",values="R2").round(3).to_string())
R.to_csv(RESULTS/"fc01_summary.csv",index=False); print("\nsaved results/fc01_summary.csv")

=== Track A (hourly) — RMSE and skill-vs-persistence ===

-- Tower 4 | RMSE --
horizon      1       6       12      24      48
model                                          
RF       134.93  142.79  140.48  142.81  143.23
XGB      135.70  143.30  140.98  146.12  144.48
clim     145.68  145.68  145.68  145.68  145.68
persist  151.25  179.16  172.73  172.49  165.79
-- Tower 4 | skill vs persistence (RF/XGB>0 = beats persistence) --
horizon     1      6      12     24     48
model                                     
RF       0.108  0.203  0.187  0.172  0.136
XGB      0.103  0.200  0.184  0.153  0.129
clim     0.037  0.187  0.157  0.155  0.121
persist  0.000  0.000  0.000  0.000  0.000

-- Tower 9 | RMSE --
horizon      1       6       12      24      48
model                                          
RF       125.82  132.63  132.14  131.93  134.00
XGB      129.94  138.35  138.80  135.02  136.01
clim     138.81  138.81  138.81  138.81  138.81
persist  138.12  170.67  176.11  163.13  174.

## 5  Append to benchmarks.csv (FC-01)

In [6]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="FC-01"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"FC-01","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"driver-conditional (AR+future-exog, EXT/SMS_MET)","track":r["track"],
        "horizon":int(r["horizon"]),"split":"fc_main" if r.tower in (4,9) else "fc_t2folds",
        "R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],"MBE":r["MBE"],"n_test":int(r["n_test"]),
        "skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],"date":today,
        "notes":"FC-01 forecasting baselines+ML; driver-conditional; train gap-filled/eval observed (D-36/D-37)"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} FC-01 rows. Total {len(comb)}.")

Wrote 108 FC-01 rows. Total 2963.
